# PolicyDelta

Semantic Privacy Policy Change Detector

Run the cells in order using **Runtime → Run all** in Google Colab.

## Configuration

In [1]:
import hashlib
import importlib.util
import json
import platform
import random
import sys
import unittest
from dataclasses import asdict, dataclass
from typing import Literal


class PolicyDeltaError(ValueError):
    """Invalid policy data or application configuration."""


class EvidenceError(PolicyDeltaError):
    """Source evidence could not be verified."""


@dataclass(frozen=True, slots=True)
class Settings:
    seed: int = 42
    max_document_chars: int = 250_000

    def __post_init__(self) -> None:
        if type(self.seed) is not int or not 0 <= self.seed < 2**32:
            raise PolicyDeltaError("Seed must be a nonnegative 32-bit integer.")
        if type(self.max_document_chars) is not int or self.max_document_chars < 1:
            raise PolicyDeltaError("Document character limit must be positive.")


SETTINGS = Settings()


def initialize(settings: Settings = SETTINGS) -> dict[str, str | int]:
    if sys.version_info < (3, 10):
        raise RuntimeError("Python 3.10 or later is required.")
    random.seed(settings.seed)
    try:
        colab = importlib.util.find_spec("google.colab") is not None
    except (ImportError, ModuleNotFoundError, ValueError):
        colab = False
    return {
        "python": platform.python_version(),
        "environment": "colab" if colab else "python",
        "seed": settings.seed,
    }


RUNTIME = initialize()
print(f"PolicyDelta | Python {RUNTIME['python']} | {RUNTIME['environment']}")

PolicyDelta | Python 3.13.16 | colab


## Document model

In [2]:
@dataclass(frozen=True, slots=True)
class PolicyDocument:
    document_id: str
    role: Literal["old", "new"]
    text: str
    sha256: str

    def __post_init__(self) -> None:
        if not isinstance(self.document_id, str) or not self.document_id.strip():
            raise PolicyDeltaError("Document ID must be nonempty.")
        if self.role not in ("old", "new"):
            raise PolicyDeltaError("Document role must be 'old' or 'new'.")
        if not isinstance(self.text, str) or not self.text.strip():
            raise PolicyDeltaError("Policy text must be nonempty.")
        if self.sha256 != hashlib.sha256(self.text.encode("utf-8")).hexdigest():
            raise EvidenceError("Document checksum does not match its content.")

    @classmethod
    def from_text(
        cls,
        text: str,
        *,
        role: Literal["old", "new"],
        document_id: str,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        if not isinstance(text, str):
            raise PolicyDeltaError("Policy content must be text.")
        if len(text) > settings.max_document_chars:
            raise PolicyDeltaError("Document exceeds the configured size limit.")
        digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
        return cls(document_id, role, text, digest)

    def excerpt(self, start: int, end: int) -> "EvidenceSpan":
        if (type(start) is not int or type(end) is not int
                or start < 0 or end > len(self.text) or start >= end):
            raise EvidenceError("Evidence offsets are outside the source text.")
        return EvidenceSpan(self.document_id, start, end, self.text[start:end])


@dataclass(frozen=True, slots=True)
class EvidenceSpan:
    document_id: str
    start: int
    end: int
    text: str


def verify_evidence(span: EvidenceSpan, document: PolicyDocument) -> None:
    """Confirm that a cited excerpt is an exact slice of the source."""
    if (span.document_id != document.document_id
            or type(span.start) is not int
            or type(span.end) is not int
            or span.start < 0
            or span.start >= span.end
            or span.end > len(document.text)
            or document.text[span.start:span.end] != span.text):
        raise EvidenceError("Evidence does not match the source document.")
    if hashlib.sha256(document.text.encode("utf-8")).hexdigest() != document.sha256:
        raise EvidenceError("Source document integrity check failed.")


def document_json(document: PolicyDocument) -> str:
    return json.dumps(asdict(document), ensure_ascii=False, indent=2)

## Validation tests

In [3]:
class PolicyDeltaTests(unittest.TestCase):
    def setUp(self) -> None:
        self.document = PolicyDocument.from_text(
            "We retain personal data for 30 days.", role="old", document_id="sample"
        )

    def test_default_settings(self):
        self.assertEqual(Settings().seed, 42)

    def test_invalid_settings(self):
        for values in ({"seed": -1}, {"seed": True}, {"max_document_chars": 0}):
            with self.subTest(values=values), self.assertRaises(PolicyDeltaError):
                Settings(**values)

    def test_document_checksum(self):
        expected = hashlib.sha256(self.document.text.encode("utf-8")).hexdigest()
        self.assertEqual(self.document.sha256, expected)

    def test_empty_document(self):
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_text("  ", role="old", document_id="empty")

    def test_input_limit(self):
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_text("long text", role="old", document_id="large",
                                     settings=Settings(max_document_chars=5))

    def test_invalid_role(self):
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_text("Valid text", role="invalid", document_id="bad")

    def test_evidence_exact_match(self):
        start = self.document.text.index("30 days")
        span = self.document.excerpt(start, start + len("30 days"))
        verify_evidence(span, self.document)
        self.assertEqual(span.text, "30 days")

    def test_tampered_evidence(self):
        span = EvidenceSpan("sample", 0, 2, "No")
        with self.assertRaises(EvidenceError):
            verify_evidence(span, self.document)

    def test_evidence_other_document(self):
        span = EvidenceSpan("another", 0, 2, "We")
        with self.assertRaises(EvidenceError):
            verify_evidence(span, self.document)

    def test_out_of_bounds(self):
        with self.assertRaises(EvidenceError):
            self.document.excerpt(0, len(self.document.text) + 1)

    def test_unicode_offsets(self):
        doc = PolicyDocument.from_text("Adatkezelés ✓", role="new", document_id="unicode")
        span = doc.excerpt(12, 13)
        self.assertEqual(span.text, "✓")
        verify_evidence(span, doc)

    def test_json_serialization(self):
        result = json.loads(document_json(self.document))
        self.assertEqual(result["role"], "old")
        self.assertEqual(result["sha256"], self.document.sha256)

    def test_reproducible_seed(self):
        initialize(Settings(seed=7))
        first = random.random()
        initialize(Settings(seed=7))
        self.assertEqual(first, random.random())
        initialize()


suite = unittest.defaultTestLoader.loadTestsFromTestCase(PolicyDeltaTests)
result = unittest.TextTestRunner(verbosity=1).run(suite)
if not result.wasSuccessful():
    raise RuntimeError("PolicyDelta validation failed.")
print(f"Validation passed: {result.testsRun} tests")

.............
----------------------------------------------------------------------
Ran 13 tests in 0.026s

OK


Validation passed: 13 tests
